In [1]:
import os
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

print("cwd", os.getcwd())
print("cuda_available", torch.cuda.is_available())
print("drive_exists", Path("/content/drive").exists())
print("mydrive_exists", Path("/content/drive/MyDrive").exists())
print("shortcut_root_exists", Path("/content/drive/.shortcut-targets-by-id").exists())

Mounted at /content/drive
cwd /content
cuda_available True
drive_exists True
mydrive_exists True
shortcut_root_exists True


In [2]:
from pathlib import Path
import json

shortcut_root = Path("/content/drive/.shortcut-targets-by-id/1bHJGtRhmrcZ8xaEG3DVnHfQhMaGnlP5_")
scan_roots = [
    shortcut_root / "trajectreview",
    Path("/content/drive/MyDrive/trajectreview"),
]
results_root_candidates = [
    Path("/content/drive/MyDrive/trajectreview/modeling"),
    shortcut_root / "trajectreview" / "modeling",
]
results_root = next((p for p in results_root_candidates if p.exists()), results_root_candidates[0])
candidate_doc_path = Path("/content/runbook_drive_candidates.json")

def infer_session_id(path: Path) -> str:
    return path.stem if path.suffix.lower() == ".zip" else path.name

zip_map = {}
dir_map = {}
for root in scan_roots:
    if not root.exists():
        continue
    for zip_path in sorted(root.rglob("*.zip")):
        stat = zip_path.stat()
        key = (infer_session_id(zip_path), stat.st_size)
        zip_map[key] = {
            "kind": "zip",
            "session_id": infer_session_id(zip_path),
            "label": f"{infer_session_id(zip_path)} [zip]",
            "path": str(zip_path),
            "size_bytes": stat.st_size,
        }
    for manifest_path in sorted(root.rglob("session_manifest.json")):
        session_dir = manifest_path.parent
        dir_map[infer_session_id(session_dir)] = {
            "kind": "dir",
            "session_id": infer_session_id(session_dir),
            "label": f"{infer_session_id(session_dir)} [dir]",
            "path": str(session_dir),
        }

candidate_doc = {
    "results_root": str(results_root),
    "candidate_count": len(zip_map) + len(dir_map),
    "candidates": sorted(list(zip_map.values()) + list(dir_map.values()), key=lambda x: (x["session_id"], x["kind"], x["path"])),
}
candidate_doc_path.write_text(json.dumps(candidate_doc, indent=2, ensure_ascii=False), encoding="utf-8")
print("candidate_doc_path", candidate_doc_path)
print("results_root", results_root)
print("candidate_count", candidate_doc["candidate_count"])
for idx, item in enumerate(candidate_doc["candidates"]):
    print(f"[{idx}] {item['label']}: {item['path']}")

candidate_doc_path /content/runbook_drive_candidates.json
results_root /content/drive/MyDrive/trajectreview/modeling
candidate_count 10
[0] debug_bundle [zip]: /content/drive/MyDrive/trajectreview/modeling/trajectreview-correcting-session-20260402-051025_da3_record_route_v01/debug_bundle.zip
[1] proof_giant_compare_bundle [zip]: /content/drive/MyDrive/trajectreview/modeling/trajectreview-correcting-session-20260402-051025_da3_record_route_v01/proof_giant_compare_bundle.zip
[2] session-20260328-103250 [zip]: /content/drive/MyDrive/trajectreview/correcting/session-20260328-103250.zip
[3] session-20260402-051025 [dir]: /content/drive/MyDrive/trajectreview/modeling/trajectreview-correcting-session-20260402-051025/session-20260402-051025
[4] trajectreview-correcting-export [zip]: /content/drive/MyDrive/trajectreview/correcting/trajectreview-correcting-export.zip
[5] trajectreview-correcting-session-20260331-034831 [zip]: /content/drive/MyDrive/trajectreview/correcting/trajectreview-correcti

In [3]:
from pathlib import Path
import json
import ipywidgets as widgets
from IPython.display import display

candidate_doc = json.loads(Path("/content/runbook_drive_candidates.json").read_text(encoding="utf-8"))
selected_doc_path = Path("/content/runbook_selected_input.json")
options = [(f"[{idx}] {item['label']}", idx) for idx, item in enumerate(candidate_doc["candidates"])]
dropdown = widgets.Dropdown(options=options, description="input", layout=widgets.Layout(width="95%"))
button = widgets.Button(description="selected input を保存", button_style="success")
output = widgets.Output()

def on_click(_):
    selected = candidate_doc["candidates"][dropdown.value]
    selected_doc = {
        "selected_index": dropdown.value,
        "kind": selected["kind"],
        "session_id": selected["session_id"],
        "label": selected["label"],
        "path": selected["path"],
        "results_root": candidate_doc["results_root"],
    }
    selected_doc_path.write_text(json.dumps(selected_doc, indent=2, ensure_ascii=False), encoding="utf-8")
    with output:
        output.clear_output()
        print(json.dumps(selected_doc, indent=2, ensure_ascii=False))
        print("selected_exists", Path(selected["path"]).exists())

button.on_click(on_click)
display(dropdown, button, output)

Dropdown(description='input', layout=Layout(width='95%'), options=(('[0] debug_bundle [zip]', 0), ('[1] proof_…

Button(button_style='success', description='selected input を保存', style=ButtonStyle())

Output()

In [4]:
from pathlib import Path
import json

selected_doc = json.loads(Path("/content/runbook_selected_input.json").read_text(encoding="utf-8"))
print("selected_path_exists", Path(selected_doc["path"]).exists(), selected_doc["path"])
print("results_root", selected_doc["results_root"])

selected_path_exists True /content/drive/MyDrive/trajectreview/correcting/trajectreview-correcting-session-20260402-051025.zip
results_root /content/drive/MyDrive/trajectreview/modeling


In [5]:
from pathlib import Path
import shutil
import subprocess

repo_root = Path("/content/Depth-Anything-3")
if repo_root.exists():
    shutil.rmtree(repo_root)
subprocess.run(["git", "clone", "https://github.com/ByteDance-Seed/Depth-Anything-3.git", str(repo_root)], check=True)
print("repo_exists", repo_root.exists(), repo_root)

repo_exists True /content/Depth-Anything-3


In [6]:
import subprocess
subprocess.run(["python", "-m", "pip", "install", "--quiet", "addict", "evo", "moviepy==1.0.3", "pygame", "pycolmap", "plyfile", "trimesh", "gsplat", "e3nn"], check=True)
print("dependency_install_ok")

dependency_install_ok


In [7]:
import sys
from pathlib import Path

repo_root = Path("/content/Depth-Anything-3")
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from depth_anything_3.api import DepthAnything3
import gsplat
import e3nn

print("depth_anything_3_import_ok", DepthAnything3)
print("gsplat_version", getattr(gsplat, "__version__", "unknown"))
print("e3nn_version", getattr(e3nn, "__version__", "unknown"))

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



depth_anything_3_import_ok <class 'depth_anything_3.api.DepthAnything3'>
gsplat_version 1.5.3
e3nn_version 0.6.0


In [8]:
from pathlib import Path
import shutil
import subprocess

repo_root = Path("/content/Depth-Anything-3")
if repo_root.exists():
    shutil.rmtree(repo_root)

subprocess.run(
    ["git", "clone", "https://github.com/ByteDance-Seed/Depth-Anything-3.git", str(repo_root)],
    check=True,
)

print((repo_root / "src" / "depth_anything_3" / "api.py").exists())


True


In [9]:
import inspect
from depth_anything_3.api import DepthAnything3

print("inference_sig", inspect.signature(DepthAnything3.inference))

inference_sig (self, image: 'list[np.ndarray | Image.Image | str]', extrinsics: 'np.ndarray | None' = None, intrinsics: 'np.ndarray | None' = None, align_to_input_ext_scale: 'bool' = True, infer_gs: 'bool' = False, use_ray_pose: 'bool' = False, ref_view_strategy: 'str' = 'saddle_balanced', render_exts: 'np.ndarray | None' = None, render_ixts: 'np.ndarray | None' = None, render_hw: 'tuple[int, int] | None' = None, process_res: 'int' = 504, process_res_method: 'str' = 'upper_bound_resize', export_dir: 'str | None' = None, export_format: 'str' = 'mini_npz', export_feat_layers: 'Sequence[int] | None' = None, conf_thresh_percentile: 'float' = 40.0, num_max_points: 'int' = 1000000, show_cameras: 'bool' = True, feat_vis_fps: 'int' = 15, export_kwargs: 'Optional[dict]' = {}) -> 'Prediction'


In [10]:
from pathlib import Path
import json
import shutil
import zipfile

selected_doc = json.loads(Path("/content/runbook_selected_input.json").read_text(encoding="utf-8"))
selected_path = Path(selected_doc["path"])
selected_kind = selected_doc["kind"]
session_id = selected_doc["session_id"]
results_root = Path(selected_doc["results_root"])

extract_root = Path("/content/trajectreview_input")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

if selected_kind == "zip":
    with zipfile.ZipFile(selected_path, "r") as zf:
        zf.extractall(extract_root)
else:
    shutil.copytree(selected_path, extract_root / selected_path.name)

session_manifest_hits = sorted(extract_root.rglob("session_manifest.json"))
if session_manifest_hits:
    session_outer = session_manifest_hits[0].parent
else:
    pkg_hits = sorted(extract_root.rglob("session_package.json"))
    assert pkg_hits, f"session_manifest.json or session_package.json not found under {extract_root}"
    session_outer = pkg_hits[0].parent.parent if pkg_hits[0].parent.name == "trajectreview" else pkg_hits[0].parent

session_root = session_outer / "trajectreview" if (session_outer / "trajectreview").exists() else session_outer

image_dir_candidates = [
    session_root / "images",
    session_root / "image",
    session_outer / "images",
    session_outer / "image",
]
source_images_dir = next((p for p in image_dir_candidates if p.exists()), None)
assert source_images_dir is not None, {"image_dir_candidates": [str(p) for p in image_dir_candidates]}

images_dir = session_root / "images"
if source_images_dir != images_dir:
    if images_dir.exists():
        shutil.rmtree(images_dir)
    shutil.copytree(source_images_dir, images_dir)

frame_record_candidates = [
    session_outer / "frame_record.jsonl",
    session_root / "frame_record.jsonl",
    session_root / "arcore_pose.jsonl",
    session_outer / "arcore_pose.jsonl",
]
frame_record_path = next((p for p in frame_record_candidates if p.exists()), None)
assert frame_record_path is not None, {"frame_record_candidates": [str(p) for p in frame_record_candidates]}

frame_pose_index_path = session_root / "frame_pose_index.csv"
probe_root = results_root / f"{session_id}_da3_record_route_v01"
proof_metric_dir = probe_root / "proof_metriclarge"
prod_metric_dir = probe_root / "prod_metriclarge"
proof_giant_dir = probe_root / "proof_giant"
world_dir = probe_root / "world_fusion_v01"
manifest_dir = probe_root / "manifests"

for p in [probe_root, proof_metric_dir, prod_metric_dir, proof_giant_dir, world_dir, manifest_dir]:
    p.mkdir(parents=True, exist_ok=True)

context_doc = {
    "session_id": session_id,
    "selected_kind": selected_kind,
    "selected_path": str(selected_path),
    "session_outer": str(session_outer),
    "session_root": str(session_root),
    "images_dir": str(images_dir),
    "frame_record_path": str(frame_record_path),
    "frame_pose_index_path": str(frame_pose_index_path),
    "probe_root": str(probe_root),
    "proof_metric_dir": str(proof_metric_dir),
    "prod_metric_dir": str(prod_metric_dir),
    "proof_giant_dir": str(proof_giant_dir),
    "world_dir": str(world_dir),
    "manifest_dir": str(manifest_dir),
}
Path("/content/runbook_session_context.json").write_text(json.dumps(context_doc, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(context_doc, indent=2, ensure_ascii=False))

{
  "session_id": "trajectreview-correcting-session-20260402-051025",
  "selected_kind": "zip",
  "selected_path": "/content/drive/MyDrive/trajectreview/correcting/trajectreview-correcting-session-20260402-051025.zip",
  "session_outer": "/content/trajectreview_input/session-20260402-051025",
  "session_root": "/content/trajectreview_input/session-20260402-051025/trajectreview",
  "images_dir": "/content/trajectreview_input/session-20260402-051025/trajectreview/images",
  "frame_record_path": "/content/trajectreview_input/session-20260402-051025/frame_record.jsonl",
  "frame_pose_index_path": "/content/trajectreview_input/session-20260402-051025/trajectreview/frame_pose_index.csv",
  "probe_root": "/content/drive/MyDrive/trajectreview/modeling/trajectreview-correcting-session-20260402-051025_da3_record_route_v01",
  "proof_metric_dir": "/content/drive/MyDrive/trajectreview/modeling/trajectreview-correcting-session-20260402-051025_da3_record_route_v01/proof_metriclarge",
  "prod_metric_

In [20]:
from pathlib import Path
import csv
import json

import imageio.v3 as iio
import numpy as np
import pandas as pd

ctx = json.loads(Path("/content/runbook_session_context.json").read_text(encoding="utf-8"))
images_dir = Path(ctx["images_dir"])
frame_record_path = Path(ctx["frame_record_path"])
manifest_dir = Path(ctx["manifest_dir"])

BLUR_THRESHOLD = 10
MIN_TRANSLATION_M = 0.5
MIN_ROTATION_DEG = 3.0
PROOF_MAX_FRAMES = 24

with frame_record_path.open("r", encoding="utf-8") as f:
    frame_records = [json.loads(line) for line in f if line.strip()]

def lap_var(image_path: Path) -> float:
    img = iio.imread(image_path)
    if img.ndim == 3:
        gray = img[..., :3].mean(axis=2).astype(np.float32)
    else:
        gray = img.astype(np.float32)
    gx = gray[:, 1:] - gray[:, :-1]
    gy = gray[1:, :] - gray[:-1, :]
    return float(np.var(gx) + np.var(gy))

def quat_to_rot(qx, qy, qz, qw):
    xx, yy, zz = qx*qx, qy*qy, qz*qz
    xy, xz, yz = qx*qy, qx*qz, qy*qz
    wx, wy, wz = qw*qx, qw*qy, qw*qz
    return np.array([
        [1 - 2*(yy + zz), 2*(xy - wz),     2*(xz + wy)],
        [2*(xy + wz),     1 - 2*(xx + zz), 2*(yz - wx)],
        [2*(xz - wy),     2*(yz + wx),     1 - 2*(xx + yy)],
    ], dtype=np.float32)

def pose_to_w2c(row):
    R_c2w = quat_to_rot(float(row.qx), float(row.qy), float(row.qz), float(row.qw))
    t_c2w = np.array([float(row.tx), float(row.ty), float(row.tz)], dtype=np.float32)
    R_w2c = R_c2w.T
    t_w2c = -R_w2c @ t_c2w
    out = np.eye(4, dtype=np.float32)
    out[:3, :3] = R_w2c
    out[:3, 3] = t_w2c
    return out

def build_K(row):
    return np.array([
        [float(row.fx), 0.0,           float(row.cx)],
        [0.0,           float(row.fy), float(row.cy)],
        [0.0,           0.0,           1.0],
    ], dtype=np.float32)

def rotation_delta_deg(R_prev, R_curr):
    cos_theta = (np.trace(R_prev.T @ R_curr) - 1.0) / 2.0
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    return float(np.degrees(np.arccos(cos_theta)))

rows = []
for rec in sorted(frame_records, key=lambda x: int(x.get("frameTimestampNs", 0))):
    image_name = str(rec.get("imageFileName", "")).strip()
    image_path = images_dir / image_name if image_name else None
    image_exists = bool(image_name) and image_path.exists()
    intr = rec.get("imageIntrinsics") or {}
    pose = rec.get("pose") or {}
    blur_score = lap_var(image_path) if image_exists else None

    rows.append({
        "session_id": rec.get("sessionId"),
        "record_index": rec.get("recordIndex"),
        "frame_timestamp_ns": rec.get("frameTimestampNs"),
        "capture_timestamp_ns": rec.get("captureTimestampNs"),
        "tracking_state": rec.get("trackingState"),
        "image_file_name": image_name,
        "image_path": str(image_path) if image_path else "",
        "image_exists": image_exists,
        "fx": intr.get("fx"),
        "fy": intr.get("fy"),
        "cx": intr.get("cx"),
        "cy": intr.get("cy"),
        "width": intr.get("width"),
        "height": intr.get("height"),
        "tx": pose.get("tx"),
        "ty": pose.get("ty"),
        "tz": pose.get("tz"),
        "qx": pose.get("qx"),
        "qy": pose.get("qy"),
        "qz": pose.get("qz"),
        "qw": pose.get("qw"),
        "blur_score": blur_score,
    })

manifest_df = pd.DataFrame(rows)
manifest_df.to_csv(manifest_dir / "input_frame_manifest.csv", index=False, encoding="utf-8")

qc_df = manifest_df.copy()
qc_df["qc_tracking_ok"] = qc_df["tracking_state"].fillna("") == "TRACKING"
qc_df["qc_image_ok"] = qc_df["image_exists"].fillna(False)
qc_df["qc_intrinsics_ok"] = qc_df[["fx", "fy", "cx", "cy", "width", "height"]].notna().all(axis=1)
qc_df["qc_pose_ok"] = qc_df[["tx", "ty", "tz", "qx", "qy", "qz", "qw"]].notna().all(axis=1)
qc_df["qc_blur_ok"] = qc_df["blur_score"].fillna(0.0) >= BLUR_THRESHOLD
qc_df["qc_pass"] = qc_df[["qc_tracking_ok", "qc_image_ok", "qc_intrinsics_ok", "qc_pose_ok", "qc_blur_ok"]].all(axis=1)
qc_df["skip_reason"] = ""
qc_df.loc[~qc_df["qc_tracking_ok"], "skip_reason"] = "tracking_not_ok"
qc_df.loc[qc_df["skip_reason"].eq("") & ~qc_df["qc_image_ok"], "skip_reason"] = "image_missing"
qc_df.loc[qc_df["skip_reason"].eq("") & ~qc_df["qc_intrinsics_ok"], "skip_reason"] = "intrinsics_missing"
qc_df.loc[qc_df["skip_reason"].eq("") & ~qc_df["qc_pose_ok"], "skip_reason"] = "pose_missing"
qc_df.loc[qc_df["skip_reason"].eq("") & ~qc_df["qc_blur_ok"], "skip_reason"] = "blur_low"
qc_df.to_csv(manifest_dir / "input_frame_qc.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

candidate_df = qc_df.loc[qc_df["qc_pass"]].copy().sort_values("frame_timestamp_ns").reset_index(drop=True)
assert len(candidate_df) >= 2, {"qc_pass_count": len(candidate_df)}

selected_rows = []
last_t = None
last_R = None

for row in candidate_df.itertuples(index=False):
    t = np.array([float(row.tx), float(row.ty), float(row.tz)], dtype=np.float32)
    R = quat_to_rot(float(row.qx), float(row.qy), float(row.qz), float(row.qw))

    if last_t is None:
        translation_m = None
        rotation_deg = None
        adopt = True
        motion_reason = "first_frame"
    else:
        translation_m = float(np.linalg.norm(t - last_t))
        rotation_deg = rotation_delta_deg(last_R, R)
        adopt = (translation_m >= MIN_TRANSLATION_M) or (rotation_deg >= MIN_ROTATION_DEG)

        if adopt:
            reasons = []
            if translation_m >= MIN_TRANSLATION_M:
                reasons.append("translation")
            if rotation_deg >= MIN_ROTATION_DEG:
                reasons.append("rotation")
            motion_reason = "+".join(reasons)
        else:
            motion_reason = "motion_small"

    selected_rows.append({
        **row._asdict(),
        "translation_from_prev_selected_m": translation_m,
        "rotation_from_prev_selected_deg": rotation_deg,
        "selected_for_da3": adopt,
        "motion_reason": motion_reason,
    })

    if adopt:
        last_t = t
        last_R = R

selection_df = pd.DataFrame(selected_rows)
selection_df.to_csv(manifest_dir / "pose_conversion_check.csv", index=False, encoding="utf-8")

prod_selected = selection_df.loc[selection_df["selected_for_da3"]].copy().reset_index(drop=True)
proof_selected = prod_selected.head(PROOF_MAX_FRAMES).copy().reset_index(drop=True)
assert len(proof_selected) >= 2, {"proof_selected_count": len(proof_selected)}

for name, df in [("proof", proof_selected), ("prod", prod_selected)]:
    Ks = np.stack([build_K(row) for row in df.itertuples(index=False)], axis=0)
    exts = np.stack([pose_to_w2c(row) for row in df.itertuples(index=False)], axis=0)
    np.save(manifest_dir / f"intrinsics_{name}.npy", Ks)
    np.save(manifest_dir / f"extrinsics_w2c_{name}.npy", exts)
    df.to_csv(manifest_dir / f"da3_input_manifest_{name}.csv", index=False, encoding="utf-8")

k_check = prod_selected[["image_file_name", "width", "height", "fx", "fy", "cx", "cy"]].copy()
k_check["resize_mode"] = "native"
k_check.to_csv(manifest_dir / "k_resize_check.csv", index=False, encoding="utf-8")

summary = {
    "frame_record_count": int(len(manifest_df)),
    "qc_pass_count": int(len(candidate_df)),
    "qc_skip_count": int((~qc_df["qc_pass"]).sum()),
    "prod_selected_count": int(len(prod_selected)),
    "proof_selected_count": int(len(proof_selected)),
    "blur_threshold": BLUR_THRESHOLD,
    "min_translation_m": MIN_TRANSLATION_M,
    "min_rotation_deg": MIN_ROTATION_DEG,
    "proof_max_frames": PROOF_MAX_FRAMES,
}
(manifest_dir / "qc_summary.json").write_text(json.dumps({
    "frame_record_count": summary["frame_record_count"],
    "qc_pass_count": summary["qc_pass_count"],
    "qc_skip_count": summary["qc_skip_count"],
}, indent=2, ensure_ascii=False), encoding="utf-8")
(manifest_dir / "da3_input_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("proof_first_images")
print(proof_selected[["record_index", "frame_timestamp_ns", "image_file_name"]].head(PROOF_MAX_FRAMES).to_string(index=False))

{
  "frame_record_count": 1118,
  "qc_pass_count": 1118,
  "qc_skip_count": 0,
  "prod_selected_count": 156,
  "proof_selected_count": 24,
  "blur_threshold": 10,
  "min_translation_m": 0.5,
  "min_rotation_deg": 3.0,
  "proof_max_frames": 24
}
proof_first_images
 record_index  frame_timestamp_ns            image_file_name
            1    1329419636842006 frame_1329419636842006.jpg
           53    1329421372475129 frame_1329421372475129.jpg
           57    1329421505534959 frame_1329421505534959.jpg
           62    1329421672223951 frame_1329421672223951.jpg
           71    1329421972525896 frame_1329421972525896.jpg
          104    1329423073510807 frame_1329423073510807.jpg
          108    1329423206968443 frame_1329423206968443.jpg
          112    1329423340400631 frame_1329423340400631.jpg
          115    1329423440457137 frame_1329423440457137.jpg
          118    1329423540574643 frame_1329423540574643.jpg
          121    1329423640621182 frame_1329423640621182.jpg
    

In [21]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from PIL import Image
from depth_anything_3.api import DepthAnything3

ctx = json.loads(Path("/content/runbook_session_context.json").read_text(encoding="utf-8"))
manifest_dir = Path(ctx["manifest_dir"])
proof_metric_dir = Path(ctx["proof_metric_dir"])
prod_metric_dir = Path(ctx["prod_metric_dir"])
world_dir = Path(ctx["world_dir"])

proof_df = pd.read_csv(manifest_dir / "da3_input_manifest_proof.csv")
proof_images = proof_df["image_path"].tolist()
proof_intrinsics = np.load(manifest_dir / "intrinsics_proof.npy")
proof_extrinsics = np.load(manifest_dir / "extrinsics_w2c_proof.npy")

prod_df = pd.read_csv(manifest_dir / "da3_input_manifest_prod.csv")
prod_images = prod_df["image_path"].tolist()
prod_intrinsics = np.load(manifest_dir / "intrinsics_prod.npy")
prod_extrinsics = np.load(manifest_dir / "extrinsics_w2c_prod.npy")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DepthAnything3.from_pretrained("depth-anything/DA3METRIC-LARGE").to(device=device)
proof_prediction = model.inference(
    image=proof_images,
    infer_gs=False,
    process_res=504,
    export_dir=str(proof_metric_dir),
    export_format="mini_npz-depth_vis",
)
prod_prediction = model.inference(
    image=prod_images,
    infer_gs=False,
    process_res=504,
    export_dir=str(prod_metric_dir),
    export_format="mini_npz-depth_vis",
)

depths = np.asarray(prod_prediction.depth)
assert depths is not None and len(depths) == len(prod_df), {
    "depth_count": None if depths is None else len(depths),
    "prod_selected_count": len(prod_df),
}
all_points = []
per_frame = []
stride = 24
for idx, row in enumerate(prod_df.itertuples(index=False)):
    depth = np.asarray(depths[idx]).astype(np.float32)
    K = prod_intrinsics[idx]
    w2c = prod_extrinsics[idx]
    c2w = np.linalg.inv(w2c)
    h, w = depth.shape
    grid_y, grid_x = np.mgrid[0:h:stride, 0:w:stride]
    z = depth[grid_y, grid_x]
    valid = np.isfinite(z) & (z > 0.0)
    if not np.any(valid):
        continue
    px = grid_x[valid].astype(np.float32)
    py = grid_y[valid].astype(np.float32)
    zz = z[valid].astype(np.float32)
    x = (px - K[0, 2]) * zz / K[0, 0]
    y = (py - K[1, 2]) * zz / K[1, 1]
    cam = np.stack([x, y, zz], axis=-1)
    cam_h = np.concatenate([cam, np.ones((len(cam), 1), dtype=np.float32)], axis=1)
    world = (c2w @ cam_h.T).T[:, :3]
    all_points.append(world)
    per_frame.append({"image_file_name": row.image_file_name, "point_count": int(len(world))})

assert all_points, "no world points generated"
merged = np.concatenate(all_points, axis=0).astype(np.float32)
np.save(world_dir / "world_points_multiframe.npy", merged)

with (world_dir / "world_points_multiframe.ply").open("w", encoding="utf-8") as f:
    f.write("ply\nformat ascii 1.0\n")
    f.write(f"element vertex {len(merged)}\n")
    f.write("property float x\nproperty float y\nproperty float z\n")
    f.write("end_header\n")
    for p in merged:
        f.write(f"{p[0]} {p[1]} {p[2]}\n")

sample = merged[::4] if len(merged) > 4000 else merged
mins = sample.min(axis=0)
maxs = sample.max(axis=0)
norm = (sample - mins) / np.maximum(maxs - mins, 1e-6)
preview = np.zeros((800, 800, 3), dtype=np.uint8)
px = np.clip((norm[:, 0] * 799).astype(int), 0, 799)
py = np.clip((norm[:, 1] * 799).astype(int), 0, 799)
preview[799 - py, px] = 255
Image.fromarray(preview).save(world_dir / "world_points_multiframe_preview.png")

proof_summary = {
    "route": "MetricLarge-proof",
    "image_count": len(proof_images),
    "proof_metric_dir": str(proof_metric_dir),
    "prediction_type": str(type(proof_prediction).__name__),
    "da3_camera_input_mode": "image_only",
}
prod_summary = {
    "route": "MetricLarge-production",
    "image_count": len(prod_images),
    "prod_metric_dir": str(prod_metric_dir),
    "prediction_type": str(type(prod_prediction).__name__),
    "da3_camera_input_mode": "image_only",
}
world_summary = {
    "route": "MetricLarge-production-world",
    "processed_frames": len(per_frame),
    "total_points": int(len(merged)),
    "stride": stride,
    "depth_source": "prod_prediction.depth",
    "world_projection_input_mode": "frame_record_intrinsics_and_pose",
}
(proof_metric_dir / "export_summary.json").write_text(json.dumps(proof_summary, indent=2, ensure_ascii=False), encoding="utf-8")
(prod_metric_dir / "export_summary.json").write_text(json.dumps(prod_summary, indent=2, ensure_ascii=False), encoding="utf-8")
(world_dir / "export_summary.json").write_text(json.dumps(world_summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({
    "proof_image_count": proof_summary["image_count"],
    "prod_image_count": prod_summary["image_count"],
    "processed_frames": world_summary["processed_frames"],
    "total_points": world_summary["total_points"],
}, indent=2, ensure_ascii=False))

[INFO ] using MLP layer as FFN
[INFO ] Processed Images Done taking 0.07338523864746094 seconds. Shape:  torch.Size([24, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.25312089920043945 seconds
[INFO ] Conversion to Prediction Done. Time: 0.00995945930480957 seconds
[INFO ] Export Results Done. Time: 0.4565598964691162 seconds
[INFO ] Processed Images Done taking 0.49935078620910645 seconds. Shape:  torch.Size([156, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 1.4010505676269531 seconds
[INFO ] Conversion to Prediction Done. Time: 0.17164301872253418 seconds
[INFO ] Export Results Done. Time: 3.009833812713623 seconds
{
  "proof_image_count": 24,
  "prod_image_count": 156,
  "processed_frames": 156,
  "total_points": 52416
}


In [13]:
from pathlib import Path
import json
import shutil
import zipfile

selected_doc = json.loads(Path("/content/runbook_selected_input.json").read_text(encoding="utf-8"))
selected_path = Path(selected_doc["path"])
selected_kind = selected_doc["kind"]
session_id = selected_doc["session_id"]
results_root = Path(selected_doc["results_root"])

extract_root = Path("/content/trajectreview_input")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

if selected_kind == "zip":
    with zipfile.ZipFile(selected_path, "r") as zf:
        zf.extractall(extract_root)
else:
    shutil.copytree(selected_path, extract_root / selected_path.name)

session_manifest_hits = sorted(extract_root.rglob("session_manifest.json"))
if session_manifest_hits:
    session_outer = session_manifest_hits[0].parent
else:
    pkg_hits = sorted(extract_root.rglob("session_package.json"))
    assert pkg_hits, f"session_manifest.json or session_package.json not found under {extract_root}"
    session_outer = pkg_hits[0].parent.parent if pkg_hits[0].parent.name == "trajectreview" else pkg_hits[0].parent

session_root = session_outer / "trajectreview" if (session_outer / "trajectreview").exists() else session_outer

image_dir_candidates = [
    session_root / "images",
    session_root / "image",
    session_outer / "images",
    session_outer / "image",
]
source_images_dir = next((p for p in image_dir_candidates if p.exists()), None)
assert source_images_dir is not None, {"image_dir_candidates": [str(p) for p in image_dir_candidates]}

images_dir = session_root / "images"
if source_images_dir != images_dir:
    if images_dir.exists():
        shutil.rmtree(images_dir)
    shutil.copytree(source_images_dir, images_dir)

frame_record_candidates = [
    session_outer / "frame_record.jsonl",
    session_root / "frame_record.jsonl",
    session_root / "arcore_pose.jsonl",
    session_outer / "arcore_pose.jsonl",
]
frame_record_path = next((p for p in frame_record_candidates if p.exists()), None)
assert frame_record_path is not None, {"frame_record_candidates": [str(p) for p in frame_record_candidates]}

frame_pose_index_path = session_root / "frame_pose_index.csv"
probe_root = results_root / f"{session_id}_da3_record_route_v01"
proof_metric_dir = probe_root / "proof_metriclarge"
prod_metric_dir = probe_root / "prod_metriclarge"
proof_giant_dir = probe_root / "proof_giant"
world_dir = probe_root / "world_fusion_v01"
manifest_dir = probe_root / "manifests"

for p in [probe_root, proof_metric_dir, prod_metric_dir, proof_giant_dir, world_dir, manifest_dir]:
    p.mkdir(parents=True, exist_ok=True)

context_doc = {
    "session_id": session_id,
    "selected_kind": selected_kind,
    "selected_path": str(selected_path),
    "session_outer": str(session_outer),
    "session_root": str(session_root),
    "images_dir": str(images_dir),
    "frame_record_path": str(frame_record_path),
    "frame_pose_index_path": str(frame_pose_index_path),
    "probe_root": str(probe_root),
    "proof_metric_dir": str(proof_metric_dir),
    "prod_metric_dir": str(prod_metric_dir),
    "proof_giant_dir": str(proof_giant_dir),
    "world_dir": str(world_dir),
    "manifest_dir": str(manifest_dir),
    "camera_pose_source": "da3_estimated",
}
Path("/content/runbook_session_context.json").write_text(json.dumps(context_doc, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(context_doc, indent=2, ensure_ascii=False))

{
  "session_id": "trajectreview-correcting-session-20260402-051025",
  "selected_kind": "zip",
  "selected_path": "/content/drive/MyDrive/trajectreview/correcting/trajectreview-correcting-session-20260402-051025.zip",
  "session_outer": "/content/trajectreview_input/session-20260402-051025",
  "session_root": "/content/trajectreview_input/session-20260402-051025/trajectreview",
  "images_dir": "/content/trajectreview_input/session-20260402-051025/trajectreview/images",
  "frame_record_path": "/content/trajectreview_input/session-20260402-051025/frame_record.jsonl",
  "frame_pose_index_path": "/content/trajectreview_input/session-20260402-051025/trajectreview/frame_pose_index.csv",
  "probe_root": "/content/drive/MyDrive/trajectreview/modeling/trajectreview-correcting-session-20260402-051025_da3_record_route_v01",
  "proof_metric_dir": "/content/drive/MyDrive/trajectreview/modeling/trajectreview-correcting-session-20260402-051025_da3_record_route_v01/proof_metriclarge",
  "prod_metric_

In [16]:
from pathlib import Path
import csv
import json

import imageio.v3 as iio
import numpy as np
import pandas as pd

ctx = json.loads(Path("/content/runbook_session_context.json").read_text(encoding="utf-8"))
images_dir = Path(ctx["images_dir"])
frame_record_path = Path(ctx["frame_record_path"])
manifest_dir = Path(ctx["manifest_dir"])

BLUR_THRESHOLD = 8.0
MIN_FRAME_GAP = 3
PROOF_MAX_FRAMES = 24

with frame_record_path.open("r", encoding="utf-8") as f:
    frame_records = [json.loads(line) for line in f if line.strip()]

def lap_var(image_path: Path) -> float:
    img = iio.imread(image_path)
    if img.ndim == 3:
        gray = img[..., :3].mean(axis=2).astype(np.float32)
    else:
        gray = img.astype(np.float32)
    gx = gray[:, 1:] - gray[:, :-1]
    gy = gray[1:, :] - gray[:-1, :]
    return float(np.var(gx) + np.var(gy))

rows = []
for rec in sorted(frame_records, key=lambda x: (int(x.get("frameTimestampNs", 0)), int(x.get("recordIndex", 0) or 0))):
    image_name = str(rec.get("imageFileName", "")).strip()
    image_path = images_dir / image_name if image_name else None
    image_exists = bool(image_name) and image_path.exists()
    intr = rec.get("imageIntrinsics") or {}
    pose = rec.get("pose") or {}
    blur_score = lap_var(image_path) if image_exists else None

    rows.append({
        "session_id": rec.get("sessionId"),
        "record_index": rec.get("recordIndex"),
        "frame_timestamp_ns": rec.get("frameTimestampNs"),
        "capture_timestamp_ns": rec.get("captureTimestampNs"),
        "tracking_state": rec.get("trackingState"),
        "image_file_name": image_name,
        "image_path": str(image_path) if image_path else "",
        "image_exists": image_exists,
        "fx": intr.get("fx"),
        "fy": intr.get("fy"),
        "cx": intr.get("cx"),
        "cy": intr.get("cy"),
        "width": intr.get("width"),
        "height": intr.get("height"),
        "tx": pose.get("tx"),
        "ty": pose.get("ty"),
        "tz": pose.get("tz"),
        "qx": pose.get("qx"),
        "qy": pose.get("qy"),
        "qz": pose.get("qz"),
        "qw": pose.get("qw"),
        "blur_score": blur_score,
    })

manifest_df = pd.DataFrame(rows)

# 1レコード1画像化
manifest_df = (
    manifest_df.sort_values(["frame_timestamp_ns", "record_index"])
    .drop_duplicates(subset=["image_file_name"], keep="first")
    .reset_index(drop=True)
)

manifest_df.to_csv(manifest_dir / "input_frame_manifest.csv", index=False, encoding="utf-8")

qc_df = manifest_df.copy()
qc_df["qc_tracking_ok"] = qc_df["tracking_state"].fillna("").isin(["TRACKING", ""])
qc_df["qc_image_ok"] = qc_df["image_exists"].fillna(False)
qc_df["qc_intrinsics_ok"] = qc_df[["fx", "fy", "cx", "cy", "width", "height"]].notna().all(axis=1)
qc_df["qc_blur_ok"] = qc_df["blur_score"].fillna(0.0) >= BLUR_THRESHOLD

# DA3推定pose版では ARCore pose 必須にしない
qc_df["qc_pose_ok"] = qc_df[["tx", "ty", "tz", "qx", "qy", "qz", "qw"]].notna().all(axis=1)
qc_df["qc_pass"] = qc_df[["qc_tracking_ok", "qc_image_ok", "qc_intrinsics_ok", "qc_blur_ok"]].all(axis=1)

qc_df["skip_reason"] = ""
qc_df.loc[~qc_df["qc_tracking_ok"], "skip_reason"] = "tracking_not_ok"
qc_df.loc[qc_df["skip_reason"].eq("") & ~qc_df["qc_image_ok"], "skip_reason"] = "image_missing"
qc_df.loc[qc_df["skip_reason"].eq("") & ~qc_df["qc_intrinsics_ok"], "skip_reason"] = "intrinsics_missing"
qc_df.loc[qc_df["skip_reason"].eq("") & ~qc_df["qc_blur_ok"], "skip_reason"] = "blur_low"

qc_df.to_csv(manifest_dir / "input_frame_qc.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

def build_K(row):
    return np.array([
        [float(row.fx), 0.0, float(row.cx)],
        [0.0, float(row.fy), float(row.cy)],
        [0.0, 0.0, 1.0],
    ], dtype=np.float32)

candidate_df = qc_df.loc[qc_df["qc_pass"]].copy().sort_values("frame_timestamp_ns").reset_index(drop=True)
assert len(candidate_df) >= 2, {"qc_pass_count": len(candidate_df)}

selected_rows = []
last_adopted_idx = None
for idx, row in enumerate(candidate_df.itertuples(index=False)):
    if last_adopted_idx is None:
        adopt = True
        skip_reason = ""
        frame_gap_from_prev_selected = None
    else:
        frame_gap_from_prev_selected = idx - last_adopted_idx
        adopt = frame_gap_from_prev_selected >= MIN_FRAME_GAP
        skip_reason = "" if adopt else "frame_gap_small"

    selected_rows.append({
        **row._asdict(),
        "frame_gap_from_prev_selected": frame_gap_from_prev_selected,
        "prod_adopted": adopt,
        "prod_skip_reason": skip_reason,
    })

    if adopt:
        last_adopted_idx = idx

prod_df = pd.DataFrame(selected_rows)
prod_df.to_csv(manifest_dir / "pose_conversion_check.csv", index=False, encoding="utf-8")

prod_selected = prod_df.loc[prod_df["prod_adopted"]].copy().reset_index(drop=True)
proof_selected = prod_selected.head(min(PROOF_MAX_FRAMES, len(prod_selected))).copy()
assert len(proof_selected) >= 2, {"proof_selected": len(proof_selected)}

for name, df in [("proof", proof_selected), ("prod", prod_selected)]:
    Ks = np.stack([build_K(row) for row in df.itertuples(index=False)], axis=0)
    np.save(manifest_dir / f"intrinsics_{name}.npy", Ks)
    df.to_csv(manifest_dir / f"da3_input_manifest_{name}.csv", index=False, encoding="utf-8")

k_check = prod_selected[["image_file_name", "width", "height", "fx", "fy", "cx", "cy"]].copy()
k_check["resize_mode"] = "native"
k_check.to_csv(manifest_dir / "k_resize_check.csv", index=False, encoding="utf-8")

summary = {
    "frame_record_count": int(len(manifest_df)),
    "qc_pass_count": int(len(candidate_df)),
    "qc_skip_count": int((~qc_df["qc_pass"]).sum()),
    "prod_selected_count": int(len(prod_selected)),
    "proof_selected_count": int(len(proof_selected)),
    "pose_source": "da3_estimated",
    "selection_rule": f"qc + blur>={BLUR_THRESHOLD} + min_frame_gap={MIN_FRAME_GAP}",
}
(manifest_dir / "qc_summary.json").write_text(json.dumps({
    "frame_record_count": summary["frame_record_count"],
    "qc_pass_count": summary["qc_pass_count"],
    "qc_skip_count": summary["qc_skip_count"],
}, indent=2, ensure_ascii=False), encoding="utf-8")
(manifest_dir / "da3_input_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "frame_record_count": 1118,
  "qc_pass_count": 1118,
  "qc_skip_count": 0,
  "prod_selected_count": 373,
  "proof_selected_count": 36,
  "pose_source": "da3_estimated",
  "selection_rule": "qc + blur>=8.0 + min_frame_gap=3"
}


In [17]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from depth_anything_3.api import DepthAnything3

ctx = json.loads(Path("/content/runbook_session_context.json").read_text(encoding="utf-8"))
manifest_dir = Path(ctx["manifest_dir"])
proof_metric_dir = Path(ctx["proof_metric_dir"])
prod_metric_dir = Path(ctx["prod_metric_dir"])

proof_df = pd.read_csv(manifest_dir / "da3_input_manifest_proof.csv")
proof_images = proof_df["image_path"].tolist()

prod_df = pd.read_csv(manifest_dir / "da3_input_manifest_prod.csv")
prod_images = prod_df["image_path"].tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DepthAnything3.from_pretrained("depth-anything/DA3METRIC-LARGE").to(device=device)

# 深度専用。pose/intrinsics取得はここでは狙わない
proof_prediction = model.inference(
    image=proof_images,
    infer_gs=False,
    process_res=504,
    export_dir=str(proof_metric_dir),
    export_format="mini_npz-depth_vis",
)
prod_prediction = model.inference(
    image=prod_images,
    infer_gs=False,
    process_res=504,
    export_dir=str(prod_metric_dir),
    export_format="mini_npz-depth_vis",
)

depths = np.asarray(prod_prediction.depth)
assert depths is not None and len(depths) == len(prod_df), {
    "depth_count": None if depths is None else len(depths),
    "prod_selected_count": len(prod_df),
}

proof_summary = {
    "route": "MetricLarge-proof",
    "image_count": len(proof_images),
    "proof_metric_dir": str(proof_metric_dir),
    "prediction_type": str(type(proof_prediction).__name__),
    "da3_camera_input_mode": "image_only",
    "camera_pose_source": "not_used_here",
}
prod_summary = {
    "route": "MetricLarge-production",
    "image_count": len(prod_images),
    "prod_metric_dir": str(prod_metric_dir),
    "prediction_type": str(type(prod_prediction).__name__),
    "da3_camera_input_mode": "image_only",
    "camera_pose_source": "not_used_here",
}

(proof_metric_dir / "export_summary.json").write_text(json.dumps(proof_summary, indent=2, ensure_ascii=False), encoding="utf-8")
(prod_metric_dir / "export_summary.json").write_text(json.dumps(prod_summary, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps({
    "proof_image_count": proof_summary["image_count"],
    "prod_image_count": prod_summary["image_count"],
}, indent=2, ensure_ascii=False))

[INFO ] using MLP layer as FFN
[INFO ] Processed Images Done taking 0.10655093193054199 seconds. Shape:  torch.Size([36, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.3628220558166504 seconds
[INFO ] Conversion to Prediction Done. Time: 0.03276968002319336 seconds
[INFO ] Export Results Done. Time: 0.6850864887237549 seconds
[INFO ] Processed Images Done taking 1.2659006118774414 seconds. Shape:  torch.Size([373, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 3.3269805908203125 seconds
[INFO ] Conversion to Prediction Done. Time: 0.40663957595825195 seconds
[INFO ] Export Results Done. Time: 65.90701341629028 seconds
{
  "proof_image_count": 36,
  "prod_image_count": 373
}


In [18]:
from pathlib import Path
import json
import sys
import shutil

import numpy as np
import pandas as pd
import torch
from PIL import Image

for name in list(sys.modules.keys()):
    if name.startswith("depth_anything_3"):
        del sys.modules[name]

repo_root = Path("/content/Depth-Anything-3")
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from depth_anything_3.api import DepthAnything3

ctx = json.loads(Path("/content/runbook_session_context.json").read_text(encoding="utf-8"))
manifest_dir = Path(ctx["manifest_dir"])
proof_giant_dir = Path(ctx["proof_giant_dir"])

proof_df = pd.read_csv(manifest_dir / "da3_input_manifest_proof.csv")
proof_images = proof_df["image_path"].tolist()

giant_debug_dir = proof_giant_dir / "debug_da3_estimated_no_gs"
if giant_debug_dir.exists():
    shutil.rmtree(giant_debug_dir)
giant_debug_dir.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DepthAnything3.from_pretrained("depth-anything/DA3NESTED-GIANT-LARGE").to(device=device)

prediction = model.inference(
    image=proof_images,
    infer_gs=False,
    process_res=504,
    export_dir=str(giant_debug_dir),
    export_format="mini_npz-depth_vis",
)

def to_np(x):
    if x is None:
        return None
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def summarize_array(name, x):
    print(f"\n## {name}")
    if x is None:
        print("None")
        return
    arr = to_np(x)
    print("shape:", getattr(arr, "shape", None), "dtype:", getattr(arr, "dtype", None))
    if isinstance(arr, np.ndarray) and np.issubdtype(arr.dtype, np.number):
        finite = np.isfinite(arr)
        print("finite:", int(finite.sum()), "/", arr.size)
        if finite.any():
            af = arr[finite]
            print("min:", float(af.min()))
            print("max:", float(af.max()))
            print("mean:", float(af.mean()))

cand_names = [
    "depth", "depths",
    "confidence", "conf", "confidences",
    "intrinsics", "pred_intrinsics",
    "extrinsics", "pred_extrinsics",
    "camera_pose", "camera_poses",
    "mask", "valid_mask",
]

for n in cand_names:
    if hasattr(prediction, n):
        summarize_array(n, getattr(prediction, n))

depth = None
for n in ["depth", "depths"]:
    if hasattr(prediction, n):
        depth = to_np(getattr(prediction, n))
        depth_name = n
        break

conf = None
for n in ["confidence", "conf", "confidences"]:
    if hasattr(prediction, n):
        conf = to_np(getattr(prediction, n))
        conf_name = n
        break

mask = None
for n in ["mask", "valid_mask"]:
    if hasattr(prediction, n):
        mask = to_np(getattr(prediction, n))
        mask_name = n
        break

assert depth is not None, "depth/depths not found"

d0 = depth[0] if depth.ndim >= 3 else depth
c0 = conf[0] if conf is not None and conf.ndim >= 3 else conf
m0 = mask[0] if mask is not None and mask.ndim >= 3 else mask

valid = np.isfinite(d0) & (d0 > 0)
conf_p40 = None

if c0 is not None:
    cvalid = np.isfinite(c0)
    valid &= cvalid
    if cvalid.any():
        conf_p40 = float(np.percentile(c0[cvalid], 40.0))
        valid &= (c0 >= conf_p40)

if m0 is not None:
    valid &= (m0 > 0)

valid_depth = d0[valid]
print("\ndepth_name =", depth_name)
print("conf_name =", None if conf is None else conf_name)
print("mask_name =", None if mask is None else mask_name)
print("frame0 depth finite =", int(np.isfinite(d0).sum()), "/", d0.size)
print("frame0 depth positive =", int((d0 > 0).sum()))
print("frame0 conf_p40 =", conf_p40)
print("frame0 valid_depth_count =", int(valid_depth.size))
if valid_depth.size > 0:
    print("frame0 valid_depth_minmax =", float(valid_depth.min()), float(valid_depth.max()))

def build_K(row):
    return np.array([
        [float(row.fx), 0.0, float(row.cx)],
        [0.0, float(row.fy), float(row.cy)],
        [0.0, 0.0, 1.0],
    ], dtype=np.float32)

def backproject_depth_to_xyz(depth2d, K):
    h, w = depth2d.shape
    ys, xs = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    z = depth2d.astype(np.float32)
    x = (xs.astype(np.float32) - float(K[0, 2])) * z / float(K[0, 0])
    y = (ys.astype(np.float32) - float(K[1, 2])) * z / float(K[1, 1])
    return np.stack([x, y, z], axis=-1)

def write_simple_ply_xyzrgb(path, xyz, rgb, mask=None):
    xyz = np.asarray(xyz).reshape(-1, 3)
    rgb = np.asarray(rgb).reshape(-1, 3)
    finite_mask = np.isfinite(xyz).all(axis=1)
    use = finite_mask if mask is None else (finite_mask & np.asarray(mask).reshape(-1).astype(bool))
    pts = xyz[use]
    cols = np.clip(rgb[use], 0, 255).astype(np.uint8)

    with open(path, "w", encoding="utf-8") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {len(pts)}\n")
        f.write("property float x\nproperty float y\nproperty float z\n")
        f.write("property uchar red\nproperty uchar green\nproperty uchar blue\n")
        f.write("end_header\n")
        for p, c in zip(pts, cols):
            f.write(f"{p[0]} {p[1]} {p[2]} {int(c[0])} {int(c[1])} {int(c[2])}\n")

K0 = build_K(proof_df.iloc[0])

img0_full = np.array(Image.open(proof_images[0]).convert("RGB"))
h_d, w_d = d0.shape

# depth shape に合わせて RGB を縮小
img0 = np.array(
    Image.fromarray(img0_full).resize((w_d, h_d), resample=Image.BILINEAR)
)

xyz0 = backproject_depth_to_xyz(d0, K0)

simple_ply_dir = giant_debug_dir / "simple_points"
simple_ply_dir.mkdir(parents=True, exist_ok=True)

mask0 = np.isfinite(d0) & (d0 > 0)

print("img0_full shape =", img0_full.shape)
print("img0_resized shape =", img0.shape)
print("depth shape =", d0.shape)
print("xyz0 shape =", xyz0.shape)
print("mask0 shape =", mask0.shape)

write_simple_ply_xyzrgb(
    simple_ply_dir / "frame0_simple_points_all_positive.ply",
    xyz0,
    img0,
    mask0,
)

for pct in [0, 10, 20, 40]:
    v = np.isfinite(d0) & (d0 > 0)
    th = None
    if c0 is not None:
        cvalid = np.isfinite(c0)
        v &= cvalid
        if cvalid.any():
            th = float(np.percentile(c0[cvalid], pct))
            v &= (c0 >= th)
    if m0 is not None:
        v &= (m0 > 0)
    write_simple_ply_xyzrgb(simple_ply_dir / f"frame0_simple_points_confpct_{pct}.ply", xyz0, img0, v)
    print("confpct", pct, "threshold", th, "count", int(v.sum()))

summary = {
    "route": "Giant-proof-da3-estimated-debug-no-gs",
    "image_count": len(proof_images),
    "proof_giant_dir": str(proof_giant_dir),
    "debug_dir": str(giant_debug_dir),
    "prediction_type": str(type(prediction).__name__),
    "camera_pose_source": "da3_estimated",
    "has_pred_intrinsics": bool(hasattr(prediction, "pred_intrinsics") or hasattr(prediction, "intrinsics")),
    "has_pred_extrinsics": bool(hasattr(prediction, "pred_extrinsics") or hasattr(prediction, "extrinsics")),
    "frame0_valid_depth_count": int(valid_depth.size),
}
(giant_debug_dir / "export_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))

Error registering eval resolver: resolver 'eval' is already registered


config.json: 0.00B [00:00, ?B/s]

[INFO ] using SwiGLU layer as FFN
[INFO ] using MLP layer as FFN


model.safetensors:   0%|          | 0.00/6.76G [00:00<?, ?B/s]

[INFO ] Processed Images Done taking 0.12405800819396973 seconds. Shape:  torch.Size([36, 3, 378, 504])
[INFO ] Selecting reference view using strategy: saddle_balanced
[INFO ] Model Forward Pass Done. Time: 2.493488311767578 seconds
[INFO ] Conversion to Prediction Done. Time: 0.011606454849243164 seconds
[INFO ] Export Results Done. Time: 0.6668651103973389 seconds

## depth
shape: (36, 378, 504) dtype: float32
finite: 6858432 / 6858432
min: 0.9975457191467285
max: 4.7718892097473145
mean: 2.42309308052063

## conf
shape: (36, 378, 504) dtype: float32
finite: 6858432 / 6858432
min: 1.0
max: 8.48690414428711
mean: 3.2453818321228027

## intrinsics
shape: (36, 3, 3) dtype: float32
finite: 324 / 324
min: 0.0
max: 346.578857421875
mean: 125.78511047363281

## extrinsics
shape: (36, 3, 4) dtype: float32
finite: 432 / 432
min: -0.30926334857940674
max: 1.0
mean: 0.25109535455703735

depth_name = depth
conf_name = conf
mask_name = None
frame0 depth finite = 190512 / 190512
frame0 depth posi

高画質化実験用

In [19]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch

for name in list(sys.modules.keys()):
    if name.startswith("depth_anything_3"):
        del sys.modules[name]

repo_root = Path("/content/Depth-Anything-3")
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from depth_anything_3.api import DepthAnything3

ctx = json.loads(Path("/content/runbook_session_context.json").read_text(encoding="utf-8"))
manifest_dir = Path(ctx["manifest_dir"])
proof_giant_dir = Path(ctx["proof_giant_dir"])

proof_df = pd.read_csv(manifest_dir / "da3_input_manifest_proof.csv")
proof_images = proof_df["image_path"].tolist()
proof_intrinsics = np.load(manifest_dir / "intrinsics_proof.npy")
proof_extrinsics = np.load(manifest_dir / "extrinsics_w2c_proof.npy")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 元:
# model = DepthAnything3(model_name="da3-giant").to(device)
# 変更:
model = DepthAnything3.from_pretrained("depth-anything/DA3NESTED-GIANT-LARGE-1.1").to(device=device)

# 元:
# process_res=504
# num_max_points=1000000,
# 変更候補:
PROCESS_RES = 1008   # 504 / 756 / 1008
NUM_MAX_POINTS = 1250000  # 1000000 / 2000000
CONF_THRESH_PERCENTILE = 35.0  # 40.0 / 25.0 / 20.0
REF_VIEW_STRATEGY = "middle"  # "saddle_balanced" / "middle"

prediction = model.inference(
    image=proof_images,

    # 元:
    # intrinsics=proof_intrinsics,
    # extrinsics=proof_extrinsics,

    infer_gs=True,

    # 元:
    # process_res=504,
    process_res=PROCESS_RES,

    # 元:
    # ref_view_strategy="saddle_balanced",
    ref_view_strategy=REF_VIEW_STRATEGY,

    export_dir=str(proof_giant_dir),
    export_format="npz-glb-gs_ply-gs_video",

    # 元:
    # conf_thresh_percentile=40.0,
    conf_thresh_percentile=25.0,

    # 元:
    # num_max_points=1000000,
    num_max_points=NUM_MAX_POINTS,
)

generated = []
for p in sorted(proof_giant_dir.rglob("*")):
    if p.is_file():
        generated.append({
            "relative_path": str(p.relative_to(proof_giant_dir)),
            "size_bytes": int(p.stat().st_size),
        })

generated_df = pd.DataFrame(generated)
generated_df.to_csv(proof_giant_dir / "generated_files_debug.csv", index=False, encoding="utf-8")

gs_related = generated_df[
    generated_df["relative_path"].str.contains(r"(?:^gs_|/gs_|\.glb$|\.ply$|proof_gs_input_frames\.csv)", regex=True)
].copy()
gs_related.to_csv(proof_giant_dir / "generated_gs_related_files_debug.csv", index=False, encoding="utf-8")

summary = {
    # 元:
    # "route": "Giant-proof-explicit-pose",
    "route": "Giant-proof-da3-estimated-pose",

    "image_count": len(proof_images),
    "proof_giant_dir": str(proof_giant_dir),
    "prediction_type": str(type(prediction).__name__),

    # 変更条件を明示
    # "camera_pose_source": "explicit_input",
    "camera_pose_source": "da3_estimated",

    # "model_id": "da3-giant",
    "model_id": "depth-anything/DA3NESTED-GIANT-LARGE-1.1",

    # "ref_view_strategy": "saddle_balanced",
    "ref_view_strategy": REF_VIEW_STRATEGY,

    # "process_res": 504,
    "process_res": PROCESS_RES,

    # "conf_thresh_percentile": 40.0,
    "conf_thresh_percentile": CONF_THRESH_PERCENTILE,

    # "num_max_points": 1000000,
    "num_max_points": NUM_MAX_POINTS,

    "has_pred_intrinsics": bool(hasattr(prediction, "pred_intrinsics") or hasattr(prediction, "intrinsics")),
    "has_pred_extrinsics": bool(hasattr(prediction, "pred_extrinsics") or hasattr(prediction, "extrinsics")),
    "generated_file_count": int(len(generated_df)),
    "generated_gs_related_file_count": int(len(gs_related)),
    "generated_files_manifest": str(proof_giant_dir / "generated_files_debug.csv"),
    "generated_gs_related_manifest": str(proof_giant_dir / "generated_gs_related_files_debug.csv"),
}

(proof_giant_dir / "export_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\n# gs_related_files")
print(gs_related.to_string(index=False))

Error registering eval resolver: resolver 'eval' is already registered


config.json: 0.00B [00:00, ?B/s]

[INFO ] using SwiGLU layer as FFN
[INFO ] using MLP layer as FFN


model.safetensors:   0%|          | 0.00/6.76G [00:00<?, ?B/s]

[INFO ] Processed Images Done taking 0.2855541706085205 seconds. Shape:  torch.Size([36, 3, 756, 1008])
[INFO ] Selecting reference view using strategy: middle


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.90 GiB. GPU 0 has a total capacity of 39.49 GiB of which 945.44 MiB is free. Including non-PyTorch memory, this process has 38.56 GiB memory in use. Of the allocated memory 30.74 GiB is allocated by PyTorch, and 7.32 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

実験用ここまで

In [16]:
from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
from google.colab import files
from plyfile import PlyData

ctx = json.loads(Path("/content/runbook_session_context.json").read_text(encoding="utf-8"))
probe_root = Path(ctx["probe_root"])
proof_giant_dir = Path(ctx["proof_giant_dir"])

gs_ply_path = proof_giant_dir / "gs_ply" / "0000.ply"
debug_read_dir = proof_giant_dir / "debug_gs_readback"
debug_read_dir.mkdir(parents=True, exist_ok=True)

if gs_ply_path.exists():
    with open(gs_ply_path, "rb") as f:
        head = f.read(8192).decode("latin1", errors="ignore")
    (debug_read_dir / "0000_header.txt").write_text(head, encoding="utf-8")
    print(head)

    ply = PlyData.read(str(gs_ply_path))
    v = ply["vertex"]
    names = list(v.data.dtype.names)

    rows = []
    for name in names:
        arr = np.asarray(v[name])
        rec = {
            "property": name,
            "shape": str(arr.shape),
            "dtype": str(arr.dtype),
        }
        if np.issubdtype(arr.dtype, np.number):
            finite = np.isfinite(arr)
            rec["finite_count"] = int(finite.sum())
            rec["total_count"] = int(arr.size)
            if finite.any():
                af = arr[finite]
                rec["min"] = float(af.min())
                rec["max"] = float(af.max())
                rec["mean"] = float(af.mean())
        rows.append(rec)

    stats_df = pd.DataFrame(rows)
    stats_df.to_csv(debug_read_dir / "0000_property_stats.csv", index=False, encoding="utf-8")

    xyz = np.stack([
        np.asarray(v["x"]),
        np.asarray(v["y"]),
        np.asarray(v["z"]),
    ], axis=1)
    xyz_mask = np.isfinite(xyz).all(axis=1)

    xyz_only_path = debug_read_dir / "0000_xyz_only.ply"
    with open(xyz_only_path, "w", encoding="utf-8") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {int(xyz_mask.sum())}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        f.write("end_header\n")
        for p in xyz[xyz_mask]:
            f.write(f"{p[0]} {p[1]} {p[2]}\n")

    focus_cols = [n for n in names if any(k in n.lower() for k in ["scale", "opacity", "rot", "quaternion"])]
    focus_rows = []
    for name in focus_cols:
        arr = np.asarray(v[name])
        finite = np.isfinite(arr)
        rec = {
            "property": name,
            "finite_count": int(finite.sum()),
            "total_count": int(arr.size),
        }
        if finite.any():
            af = arr[finite]
            rec["min"] = float(af.min())
            rec["max"] = float(af.max())
            rec["mean"] = float(af.mean())
        focus_rows.append(rec)

    pd.DataFrame(focus_rows).to_csv(debug_read_dir / "0000_focus_stats.csv", index=False, encoding="utf-8")

targets = [
    "manifests/input_frame_manifest.csv",
    "manifests/input_frame_qc.csv",
    "manifests/da3_input_manifest_proof.csv",
    "manifests/da3_input_manifest_prod.csv",
    "manifests/pose_conversion_check.csv",
    "manifests/k_resize_check.csv",
    "proof_metriclarge/export_summary.json",
    "prod_metriclarge/export_summary.json",
    "proof_giant/export_summary.json",
    "proof_giant/generated_files_debug.csv",
    "proof_giant/generated_gs_related_files_debug.csv",
    "proof_giant/gs_ply/0000.ply",
    "proof_giant/scene.glb",
    "proof_giant/proof_gs_input_frames.csv",
    "proof_giant/debug_da3_estimated_no_gs/export_summary.json",
    "proof_giant/debug_da3_estimated_no_gs/simple_points/frame0_simple_points_all_positive.ply",
    "proof_giant/debug_da3_estimated_no_gs/simple_points/frame0_simple_points_confpct_0.ply",
    "proof_giant/debug_da3_estimated_no_gs/simple_points/frame0_simple_points_confpct_10.ply",
    "proof_giant/debug_da3_estimated_no_gs/simple_points/frame0_simple_points_confpct_20.ply",
    "proof_giant/debug_da3_estimated_no_gs/simple_points/frame0_simple_points_confpct_40.ply",
    "proof_giant/debug_gs_readback/0000_header.txt",
    "proof_giant/debug_gs_readback/0000_property_stats.csv",
    "proof_giant/debug_gs_readback/0000_focus_stats.csv",
    "proof_giant/debug_gs_readback/0000_xyz_only.ply",
    "world_fusion_v01/world_points_multiframe.npy",
    "world_fusion_v01/world_points_multiframe.ply",
    "world_fusion_v01/world_points_multiframe_preview.png",
]

# Drive上に bundle を保存
bundle_dir_drive = probe_root / "debug_bundle"
bundle_zip_drive = probe_root / "debug_bundle.zip"

if bundle_dir_drive.exists():
    shutil.rmtree(bundle_dir_drive)
bundle_dir_drive.mkdir(parents=True, exist_ok=True)

for rel in targets:
    src = probe_root / rel
    if not src.exists():
        continue
    dst = bundle_dir_drive / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

if bundle_zip_drive.exists():
    bundle_zip_drive.unlink()
shutil.make_archive(str(bundle_zip_drive.with_suffix("")), "zip", root_dir=str(bundle_dir_drive))

print("drive_debug_read_dir =", debug_read_dir)
print("drive_bundle_dir =", bundle_dir_drive)
print("drive_bundle_zip =", bundle_zip_drive)

# 必要ならその場でローカルDLも
files.download(str(bundle_zip_drive))

ply
format binary_little_endian 1.0
element vertex 14290923
property float x
property float y
property float z
property float nx
property float ny
property float nz
property float f_dc_0
property float f_dc_1
property float f_dc_2
property float opacity
property float scale_0
property float scale_1
property float scale_2
property float rot_0
property float rot_1
property float rot_2
property float rot_3
end_header
Wº¿_sm¿¡@            z<¿'¢]¿¹*b¿ÕTÁÀñwÁiÁÀªtÀô¾¾MÔ>ö°¯½Û^?¬Â¿²y¿èZ@            ni;¿yÐV¿gX¿àßáÀ^ÁûÀµsÀbX¾Pë>¬½áC\?¾¿9ós¿5@            rV¿ATv¿êx¿­+èÀo4Áã6¥ÀîÀöè\¾xö>ºB½9ÉX?ÿÃ¿]7|¿9Ä@            ÉzH¿i9j¿Hp¿ÞãÀ÷ïÁE©Àã@À<ye¾÷ºö>O0½PX?oÊ½¿'7t¿¬o@            ?T¿æút¿^øu¿F2ÓÀ#÷ÁÎ¨ÀûzÀèv¾üó>²¨½W?Ø³¹¿NÓm¿½i	@            °@¿Í3f¿
5h¿B­À6EÁ¾dÀÞrÀþK¾óòç>àJµ½xX?e:Ã¿E/v¿lô@            M¿Ê9e¿9¤`¿£'ÀùÀ¡dpÀèãmÀ®l¾miê>9¡®½:®Z?Ns¹¿pp¿/·
?µ¼À½ÙR?ÞÂ¿A<~¿¥P@            9D¿¯å<¿<J¿;²óÀéÁÿÂÀòÑÀR(z¾ÿ´ý>ÀÒ½LÁS?QY¼¿Ãu¿
@       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ここから下はキープアライブ用

In [18]:
print("drive_debug_read_dir =", debug_read_dir)
print("drive_bundle_dir =", bundle_dir_drive)
print("drive_bundle_zip =", bundle_zip_drive)

drive_debug_read_dir = /content/drive/MyDrive/trajectreview/modeling/trajectreview-correcting-session-20260402-051025_da3_record_route_v01/proof_giant/debug_gs_readback
drive_bundle_dir = /content/drive/MyDrive/trajectreview/modeling/trajectreview-correcting-session-20260402-051025_da3_record_route_v01/debug_bundle
drive_bundle_zip = /content/drive/MyDrive/trajectreview/modeling/trajectreview-correcting-session-20260402-051025_da3_record_route_v01/debug_bundle.zip
